# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Looking at Lane 4's key fields before testing anything — the lane guide's own workflow ("core idea first") and the `auditing-signals` skill both insist on this. All four fields below are heavy-tailed, which changes how I test them below: plain correlation on raw values would be dominated by a handful of giant pages, so I'll use medians, weighted rates, or log-transforms rather than means and Pearson correlation wherever it matters.

In [5]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 140)

df = pd.read_csv('/content/content_refresh_anonymized.csv')
filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
lane4 = filtered[(filtered['avg_position'] > 0) & (filtered['avg_position'] <= 20)
                  & (filtered['impressions_90d'] >= 500)].copy()
print('Lane 4 slice:', lane4.shape)
print()

for col in ['impressions_90d', 'ctr', 'engagement_rate', 'scroll_rate']:
    print(f'{col:18} mean={lane4[col].mean():>10.3f}  median={lane4[col].median():>8.3f}  '
          f'p95={lane4[col].quantile(0.95):>10.3f}  max={lane4[col].max():>12.3f}')

print()
print(f"impressions_90d skew: raw={lane4['impressions_90d'].skew():.2f}, "
      f"after log1p={np.log1p(lane4['impressions_90d']).skew():.2f}")
print('-> heavy tail confirmed: raw skew of 8.5 drops to 0.55 after log1p. Every test below')
print('   uses medians, weighted rates, or log-scale bucketing instead of raw means where it matters.')

Lane 4 slice: (12023, 44)

impressions_90d    mean=  9880.425  median=3298.000  p95= 38004.400  max=  517715.000
ctr                mean=     0.312  median=   0.210  p95=     0.960  max=       5.430
engagement_rate    mean=     3.301  median=   0.000  p95=    14.290  max=     100.000
scroll_rate        mean=    10.529  median=   5.130  p95=    40.003  max=     200.000

impressions_90d skew: raw=8.46, after log1p=0.55
-> heavy tail confirmed: raw skew of 8.5 drops to 0.55 after log1p. Every test below
   uses medians, weighted rates, or log-scale bucketing instead of raw means where it matters.


## 2. Signal test #1 / #2 / #3 (verdict each)

Three plain-language claims, each tested with a grouped table and a floor check (no verdict from a bucket under ~50 rows, per the skill's sample-size floor).

In [6]:
# ---------- TEST 1: "longer content gets more engagement" ----------
print('=== TEST 1: word_count vs engagement_rate ===')
lane4['wc_bucket'] = pd.cut(lane4['word_count'], bins=[0, 500, 1200, 2500, 100000],
                             labels=['<500', '500-1200', '1200-2500', '2500+'])
t1 = lane4.groupby('wc_bucket', observed=True).agg(
    n=('engagement_rate', 'size'),
    median_engagement=('engagement_rate', 'median'),
    mean_engagement=('engagement_rate', 'mean'),
)
print(t1.round(3))
print()
print('n=32 in the 500-1200 bucket is below the ~50-row floor -- no verdict from that bucket alone.')
print('VERDICT: MIXED / essentially FALSE. Mean engagement is nearly flat across the two buckets')
print('large enough to trust (3.60 vs 3.42) -- word count alone does not predict engagement here.')
print()

# ---------- TEST 2: "older content decays in search position" ----------
print('=== TEST 2: content age vs avg_position ===')
lane4['age_bucket'] = pd.cut(lane4['content_age_days'], bins=[0, 180, 365, 730, 100000],
                              labels=['<6mo', '6-12mo', '1-2yr', '2yr+'])
t2 = lane4.groupby('age_bucket', observed=True).agg(n=('avg_position', 'size'),
                                                     median_position=('avg_position', 'median'))
print(t2.round(3))
print(f"(max content_age_days in this slice is {lane4['content_age_days'].max():.0f} days -- "
      f"under 2 years, so the 2yr+ bucket is empty by construction, not a data gap.)")
print()
print('VERDICT: MIXED. Position does drift worse with age (7.9 -> 8.4 -> 8.6), all buckets above the')
print('floor, but the drift is small in absolute terms -- real, but not a dramatic decay story.')
print()

# ---------- TEST 3: "higher-competition keywords get lower CTR" ----------
print('=== TEST 3: competition_level vs ctr (weighted: total clicks / total impressions) ===')
t3 = lane4.groupby('competition_level').apply(
    lambda g: pd.Series({'n': len(g), 'weighted_ctr': g['clicks_90d'].sum() / g['impressions_90d'].sum()}),
    include_groups=False
)
print(t3.round(4))
print()
print('VERDICT: OPPOSITE (partially). HIGH-competition pages show the HIGHEST weighted CTR (0.0041),')
print('not the lowest -- directly against the naive claim. The full ordering (HIGH > LOW > MEDIUM) is')
print('also not monotonic, so this is not a clean reversal either -- just clear evidence the naive')
print('story does not hold in this slice.')

=== TEST 1: word_count vs engagement_rate ===
              n  median_engagement  mean_engagement
wc_bucket                                          
500-1200     32                0.0            3.433
1200-2500  1491                0.0            3.595
2500+      6838                0.6            3.424

n=32 in the 500-1200 bucket is below the ~50-row floor -- no verdict from that bucket alone.
VERDICT: MIXED / essentially FALSE. Mean engagement is nearly flat across the two buckets
large enough to trust (3.60 vs 3.42) -- word count alone does not predict engagement here.

=== TEST 2: content age vs avg_position ===
               n  median_position
age_bucket                       
<6mo        5252              7.9
6-12mo      3992              8.4
1-2yr       2779              8.6
(max content_age_days in this slice is 557 days -- under 2 years, so the 2yr+ bucket is empty by construction, not a data gap.)

VERDICT: MIXED. Position does drift worse with age (7.9 -> 8.4 -> 8.6), all

## 3. The flag-linked test

The starter pipeline's `low_ctr_visible_page` reason code assumes a **flat** `ctr < 0.5` threshold works the same regardless of position tier. If that assumption holds, the flag rate should differ meaningfully by tier (since some tiers naturally have very different baseline CTR — see Week 1's finding that every tier's CTR std exceeds its mean). If the assumption is wrong, the flag should fire at a similarly high rate almost everywhere, telling you nothing tier-specific.

In [7]:
lane4['flagged_low_ctr'] = lane4['ctr'] < 0.5
flag_by_tier = lane4.groupby('position_tier').agg(n=('flagged_low_ctr', 'size'),
                                                    pct_flagged=('flagged_low_ctr', 'mean'))
print(flag_by_tier.round(3))
print()
print('page_3_5 has n=16, below the floor -- exclude it from the verdict on its own.')
print('VERDICT: the flag fires on 75-94% of pages in every tier with enough data to trust.')
print('That is not a rule identifying a meaningful minority -- it is close to flagging almost')
print('everyone, in every tier. The rule\'s implicit assumption (that ctr<0.5 meaningfully')
print('separates underperformers) does NOT hold: it is too blunt to be tier-aware in practice,')
print('confirming the same conclusion Week 1 reached from a different angle.')

                  n  pct_flagged
position_tier                   
page_1         7064        0.791
page_3_5         16        0.938
striking       4485        0.850
top_3           458        0.751

page_3_5 has n=16, below the floor -- exclude it from the verdict on its own.
VERDICT: the flag fires on 75-94% of pages in every tier with enough data to trust.
That is not a rule identifying a meaningful minority -- it is close to flagging almost
everyone, in every tier. The rule's implicit assumption (that ctr<0.5 meaningfully
separates underperformers) does NOT hold: it is too blunt to be tier-aware in practice,
confirming the same conclusion Week 1 reached from a different angle.


## 4. What this means in practice

A content team relying on the existing `low_ctr_visible_page` flag as-is is not getting a prioritized shortlist — they're getting a list covering nearly every visible page, in every position tier, which defeats the purpose of a limited-capacity review queue. The one signal that did hold up loosely (position drifting slightly worse with content age) is real but small, not something to act on alone. The clearest actionable takeaway: any scoring built for this lane needs to be tier-adjusted (as Week 1–2's `ctr_gap` approach already does), not a single global cutoff — and claims about competition level driving CTR down should be dropped entirely, since the data shows the opposite direction for high-competition pages.

In [8]:
# No additional query needed here -- this section is the plain-words takeaway from
# the verdicts already computed and printed above.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.